In [1]:
# Standard includes
from pathlib import Path
import json
import re
from collections import defaultdict

# LLM / AI related libraries
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.language_models import ModelProfile
import torch
device = "mps" if torch.backends.mps.is_available() else "cpu"
from langchain_ollama import ChatOllama


# Local libraries
SOURCE_DIR = Path('cwd').parent.parent / "src" / "christmas_puzzle"
from christmas_puzzle.lyrics import raw_songs
TOP_K = 25

In [2]:
# Helpers

song_tags = {
    "carol_of_the_bells":["bells", "repetitive_pattern", "choral"],
    "little_drummer_boy": ["drums", "nonsense_syllables"],
    "deck_the_halls": ["fa_la_la", "choral", "cheerful"],
}

def normalize_sounds(line: str) -> str:
    line = re.sub(r"(ding[\s,-]*dong)+", "<BELL_SOUND>", line, flags=re.I)
    line = re.sub(r"(pa\s+rum\s+pum\s+pum\s+pum)+", "<DRUM_PATTERN>", line, flags=re.I)
    line = re.sub(r"(fa\s+la\s+la[^a-z]*)+", "<FA_LA_LA>", line, flags=re.I)
    return line


def slugify(title: str) -> str:
    s = title.lower()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s

def normalize_songs(raw_songs):
    cleaned = []
    for song in raw_songs:
        title = song["title"].strip()
        lyrics = song["lyrics"] or ""

        # Split on newlines, strip whitespace, drop empty lines
        lines = [
            line.strip()
            for line in lyrics.splitlines()
            if line.strip()
        ]

        cleaned.append(
            {
                "id": slugify(title),
                "title": title,
                "lines": lines,
            }
        )
    return cleaned

songs = normalize_songs(raw_songs)
for song in songs:
    song["lines"] = [normalize_sounds(l) for l in song["lines"]]



In [3]:
len(songs)

50

### Design translator from Elizabethan to modern English

In [ ]:


translator_llm = ChatOllama(
    model="mistral:latest",
    temperature=0.1,
)

TRANSLATOR_SYSTEM = """
You rewrite faux-Elizabethan holiday song clues into simple modern English.

Goals:
- Preserve the original meaning and scene.
- Use concise, natural phrasing.
- Normalize repeated “nonsense” or sound effects into special tokens:

  - For repeated drum syllables (e.g. "pa rum pum pum pum", "rum pum pum pum"):
      use <DRUM_PATTERN>
  - For bell sounds (e.g. "ding dong", "ching ching", "ting ting", "ding a ling", "ringing bell", "tolling bell"):
      use <BELL_SOUND>
  - For "fa la la" style nonsense refrains:
      use <FA_LA_LA>

Rules:
- Do NOT invent new details.
- Keep all important objects, people, and relationships.
- If the original text already contains “pa rum pum pum pum”, “ding dong”, or “fa la la”,
  replace those phrases with the appropriate token.
""".strip()

translator_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", TRANSLATOR_SYSTEM),
        ("user", "Rewrite this clue in modern English:\n\n{hint}"),
    ]
)

translator_llm = ChatOllama(
    model="mistral:latest",
    temperature=0.1,
)

translator_chain = translator_prompt | translator_llm

def to_modern(text: str) -> str:
    return translator_chain.invoke({"hint": text}).content.strip()


### Build the song index w/ sliding lyric windows

In [5]:

WINDOW_SIZE = 2  # or 3 if your lines tend to be super short

def build_song_docs(songs, window_size: int = WINDOW_SIZE):
    docs = []
    for song in songs:
        lines = song["lines"]
        for i in range(len(lines) - window_size + 1):
            window = " / ".join(lines[i : i + window_size])
            docs.append(
                Document(
                    page_content=window,
                    metadata={
                        "song_id": song["id"],
                        "title": song["title"],
                        "start_line": i,
                        "end_line": i + window_size - 1,
                    },
                )
            )
    return docs

docs = build_song_docs(songs, window_size=WINDOW_SIZE)

embed = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
    # or any embedding model you like
)

store = FAISS.from_documents(docs, embed)
retriever = store.as_retriever(search_kwargs={"k": TOP_K})


/var/folders/2d/nd_kkrs166v10fz9h75cg5v00000gn/T/ipykernel_36554/4168102438.py:24: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embed = HuggingFaceEmbeddings(


### Build the reranker - pick the best answers from a group of suggestions

In [6]:
ranker_llm = ChatOllama(
    model="mistral:latest",
    temperature=0.0,   # as deterministic as possible
)
SYSTEM_BATCH_RANKER = """
You are matching a holiday song riddle to multiple lyric excerpts.

You will receive:
- A HINT (faux-Elizabethan paraphrase).
- A numbered LIST of lyric EXCERPTS (1, 2, 3, ...).

Decide how well each excerpt matches the SAME specific lyric idea as the hint.

Scoring (0–10):
- 0–2: Unrelated. Different scene or no clear connection.
- 3–5: Generic holiday/winter/Christmas mood only. No clear shared situation.
- 6–8: Clearly related scene or idea. Same event or image (e.g. “snow on the mountain, no footprints”), but wording may differ.
- 9–10: Very strong match. The hint is essentially a paraphrase of this lyric:
  - Same concrete situation,
  - Same key objects,
  - Same relationship between them (e.g. “bad weather outside BUT warm fire inside”).

Rules:
- Do NOT give scores above 6 if you only see vague overlap like “Christmas”, “snow”, “night”, “bells”, or “joy”.
- For hints about CONTRAST (e.g. bad outside weather vs cozy fire inside), only score 9–10 if that contrast is clearly present in the excerpt.

Token rules:
- If the HINT contains a special token (like <DRUM_PATTERN>, <BELL_SOUND>, or <FA_LA_LA>),
  then high scores (9–10) should only be given to excerpts that also contain the SAME token
  or an extremely clear equivalent sound in plain text.
- If the HINT mentions <DRUM_PATTERN> but the excerpt has no drum pattern at all,
  cap the score at 6 even if the rest of the scene is similar.
- Similarly for <BELL_SOUND> and <FA_LA_LA>.

Output:
Return ONLY valid JSON:
- Top-level key "scores"
- "scores" maps string indices like "1", "2", "3", ... to numeric scores between 0 and 10.
""".strip()


batch_ranker_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", SYSTEM_BATCH_RANKER),
        (
            "user",
            "HINT:\n{hint}\n\n"
            "EXCERPTS:\n{numbered_excerpts}\n\n"
            "JSON only:"
        ),
    ]
)

batch_reranker_llm = ChatOllama(
    model="mistral:latest",
    temperature=0.0,
)

batch_rerank_chain = batch_ranker_prompt | batch_reranker_llm


def select_windows_per_song(candidates, max_to_rerank=25, per_song_limit=3):
    """
    candidates: list of docs sorted by embedding similarity
    returns: list of docs, preserving order, but at most `per_song_limit` per song
    """
    per_song_count = defaultdict(int)
    selected = []

    for doc in candidates:
        sid = doc.metadata["song_id"]
        if per_song_count[sid] >= per_song_limit:
            continue
        selected.append(doc)
        per_song_count[sid] += 1
        if len(selected) >= max_to_rerank:
            break

    return selected



def rerank_candidates_batch(hint: str, candidates, max_to_rerank: int = 25):
    docs = select_windows_per_song(candidates, max_to_rerank=max_to_rerank, per_song_limit=3)

    lines = []
    for i, doc in enumerate(docs, start=1):
        snippet = doc.page_content.replace("\n", " / ")
        lines.append(f"{i}. {snippet}")
    numbered_excerpts = "\n".join(lines)

    raw = batch_rerank_chain.invoke(
        {"hint": hint, "numbered_excerpts": numbered_excerpts}
    ).content.strip()

    try:
        data = json.loads(raw)
        scores_dict = data.get("scores", {})
    except Exception:
        scores_dict = {}

    scored_docs = []
    for i, doc in enumerate(docs, start=1):
        score = float(scores_dict.get(str(i), 0.0))
        scored_docs.append((doc, score))

    scored_docs.sort(key=lambda x: x[1], reverse=True)
    return scored_docs



In [7]:

def aggregate_by_song(scored_docs, top_songs: int = 3):
    best_score = {}
    best_doc = {}

    for doc, score in scored_docs:
        sid = doc.metadata["song_id"]
        if sid not in best_score or score > best_score[sid]:
            best_score[sid] = score
            best_doc[sid] = doc

    ranked = sorted(best_score.items(), key=lambda x: x[1], reverse=True)

    results = []
    for sid, s in ranked[:top_songs]:
        doc = best_doc[sid]
        results.append(
            {
                "song_id": sid,
                "title": doc.metadata["title"],
                "score": s,
                "sample_window": doc.page_content,
            }
        )
    return results



### Create the song guesser

In [8]:

def guess_song(elizabethan_hint: str, k_docs: int = 10, k_songs: int = 3):
    # 1) Normalize language
    modern = to_modern(elizabethan_hint)

    # 2) Retrieve lyric windows
    candidates = retriever.invoke(modern)  # already sorted by similarity

    # 3) Rerank candidates
    scored_docs = rerank_candidates_batch(modern, candidates, TOP_K)

    # 4) Aggregate score by song -- return top_songs 
    ranked_songs = aggregate_by_song(scored_docs, top_songs = 3)

    return {
        "modern_query": modern,
        "results": ranked_songs,
    }


### Run the puzzle

In [9]:
elizabethan_hints = ["O, the winds without are dreadful, Yet the fire within maketh me joyous.",
                   "Tolling tintinnabulum, tolling tintinnabulum—Ah! The rhythm of tolling tintinnabulum! The tintinnabulum doth swing and—Yea!—also dost ring",
                   """"Come!", they didst proclaim, pa rum pum pum pum
A newly wrought monarch to behold, pa rum pum pum pum""",
"""Santa, thou sweet babe, prithee slip a furred mantle 'neath the tree for me;
I have been a most virtuous maid.""",
"""Still night, hallow'd night,
All is hush'd, all is aglow.""",
"""Cover thy pate—Lo! Chanukah approacheth,
Such mirth and excit'ment-ukah, to make merry for Chanukah.""",
"""Thou art a churlish wight—verily, a knave art thou
Thou art as huggable as a prickly plant, as winsome as a serpent of the watery deep""",
"""So this be Yuletide, and what hast thou wrought?
Another year hath ended, and a new one hath but begun.""",
"""Old Hiems, with his black and icy crown, was a blithe and merry spirit,
With a pipe of maize, and a proboscis of button wrought, and twain eyes of ember.""",
"""The snow doth gleam as white upon the mountain's height,
As all who pass on foot go untrack'd""",
"""My heart longeth for a Yuletide clad in white,
E'en as those I knew in days of yore.""",
"""'Twas the four and twentieth of December upon Hollis Avenue after night's fall,
When I espied a gentleman reclining with his hound in the village green."""]

for hint in elizabethan_hints:
    out = guess_song(hint)
    print("Hint:",hint)
    print("Modern query:", out["modern_query"])
    for r in out["results"][:1]:
        print(f"- {r['title']} (score={r['score']})")
        print(f"  Match: {r['sample_window']}")
    print("\n")

Hint: O, the winds without are dreadful, Yet the fire within maketh me joyous.
Modern query: The cold wind outside is frightening, But the warmth inside brings me happiness.
<DRUM_PATTERN> <FA_LA_LA> <BELL_SOUND> <FA_LA_LA> <BELL_SOUND> <FA_LA_LA> <FA_LA_LA>

(Original: "O wind, thou wast never kinder, Whilst the bitter sky frowns and blows; Thy teeth are chatter'd in their cold, Thine eyes are fearfully aglow. O fire, by Hearth and Candle glowing, Who quickly my poor bosom warm, And chases frosty fear away, That crept into my heart in storm.")
- Carol of the bells (score=10.0)
  Match: With joyful ring, all caroling / <BELL_SOUND>, <BELL_SOUND>


Hint: Tolling tintinnabulum, tolling tintinnabulum—Ah! The rhythm of tolling tintinnabulum! The tintinnabulum doth swing and—Yea!—also dost ring
Modern query: Ringing bell, ringing bell - Listen to the bell's rhythm!
The bell swings back and forth, and yes, it also rings.
- Ding Dong Merrily on High (score=9.0)
  Match: <BELL_SOUND>! Merrily 